## Data Cleaning Process

**Import libraries**

In [33]:
# Import library
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 

**Data Loading & Dataset Overview**

In [34]:
# Data Loading
df = pd.read_csv('IAQ Baqubah Teaching Hospital .csv')
df.head()
print("Shape: ", df.shape)
df.info()
df.describe(include='all')

Shape:  (523524, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 523524 entries, 0 to 523523
Data columns (total 12 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   CO2          523524 non-null  float64
 1   TVOC         523524 non-null  float64
 2   PM10         523524 non-null  float64
 3   PM2.5        523524 non-null  float64
 4   eCO2         203159 non-null  float64
 5   CO           523524 non-null  float64
 6   Air Quality  523524 non-null  float64
 7   LDR          523524 non-null  float64
 8   O3           523524 non-null  float64
 9   Temp         523524 non-null  float64
 10  Hum          523524 non-null  float64
 11  ts           523524 non-null  object 
dtypes: float64(11), object(1)
memory usage: 47.9+ MB


,CO2,TVOC,PM10,PM2.5,eCO2,CO,Air Quality,LDR,O3,Temp,Hum,ts
count,523524.000000,523524.000000,523524.000000,523524.000000,203159.000000,523524.000000,523524.000000,523524.000000,523524.000000,523524.000000,523524.000000,523524
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,295893
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11/4/2024 2:02
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4
mean,512.281274,50.626842,782.512225,102.466800,440.613554,409.098120,117.580437,892.647055,626.467808,26.378307,44.792178,NaN
std,185.873552,16.179725,362.372013,52.308006,113.883309,118.291014,54.061087,173.132083,61.982747,2.735585,6.022637,NaN
min,383.000000,4.000000,4.330000,0.000000,400.000000,115.800000,30.000000,14.000000,456.000000,19.360000,24.500000,NaN
25%,422.400000,46.200000,600.071500,65.106000,400.000000,326.400000,66.200000,917.600000,586.600000,24.380000,41.580000,NaN
50%,437.000000,49.000000,746.638000,111.936000,402.000000,409.600000,125.800000,932.000000,625.600000,26.000000,44.940000,NaN
75%,448.800000,54.600000,982.688000,136.362500,409.000000,489.800000,163.800000,945.800000,668.400000,28.100000,48.360000,NaN


**Handling Missing Values**

In [35]:
# Check for missing values
df.isna().sum().sort_values(ascending=False)

eCO2           320365
CO2                 0
PM10                0
TVOC                0
PM2.5               0
CO                  0
Air Quality         0
LDR                 0
O3                  0
Temp                0
Hum                 0
ts                  0
dtype: int64

In [36]:
# Check percentage of missing values
missing_percentage = (df.isna().sum() / len(df)) * 100
missing_percentage = missing_percentage[missing_percentage > 0].sort_values(ascending=False)
missing_percentage

eCO2    61.193947
dtype: float64

In [37]:
# Drop columns with more than 50% missing values
cols_to_drop = missing_percentage[missing_percentage > 50].index
df.drop(columns=cols_to_drop, inplace=True)
df.shape

(523524, 11)

**Check for duplicated rows**

In [38]:
# Check for duplicate rows
df.duplicated().sum()

np.int64(0)

**Convert data type**

In [40]:
# Convert 'timestamp' column to datetime
df['ts'] = pd.to_datetime(df['ts'], errors='coerce')
df = df.sort_values('ts').set_index('ts')


In [41]:
df_5min = df.resample('5T').mean()

C:\Users\Binh\AppData\Local\Temp\ipykernel_27860\1866487118.py:1: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_5min = df.resample('5T').mean()


In [43]:
df_5min.isna().sum()

CO2            5755
TVOC           5755
PM10           5755
PM2.5          5755
CO             5755
Air Quality    5755
LDR            5755
O3             5755
Temp           5755
Hum            5755
dtype: int64

In [50]:
iaqs = ['CO2', 'TVOC', 'PM10', 'PM2.5', 'CO', 'Air Quality', 'LDR', 'O3', 'Temp', 'Hum']  

df_5min[iaqs] = df_5min[iaqs].interpolate(
    method='time',      
    limit=3,
    limit_direction='both'
)


In [51]:
df_5min[iaqs] = df_5min[iaqs].ffill().bfill()
df_5min.isna().sum()

CO2            0
TVOC           0
PM10           0
PM2.5          0
CO             0
Air Quality    0
LDR            0
O3             0
Temp           0
Hum            0
dtype: int64

In [ ]:
df_5min.to_csv("cleaned_dataset.csv", index=True)